In [1]:
import pandas as pd
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_scenario

In [2]:
location = "india"
vehicle = "rice"

In [3]:
# Parameters
location = "india"
vehicle = "rice"


In [4]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/person_time_anemia.parquet"
if pathlib.Path(path).is_file():
    pregnancy_person_time_anemia = pd.read_parquet(path)
else:
    pregnancy_person_time_anemia = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/person_time_anemia.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_person_time_anemia

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,person_time,impairment,anemia,not_anemic,10_to_14,invalid,1,baseline,0,2,176.044657
1,person_time,impairment,anemia,not_anemic,10_to_14,invalid,2,baseline,0,2,112.447373
2,person_time,impairment,anemia,not_anemic,10_to_14,invalid,3,baseline,0,2,139.176666
3,person_time,impairment,anemia,not_anemic,10_to_14,invalid,4,baseline,0,2,150.237064
4,person_time,impairment,anemia,not_anemic,10_to_14,invalid,5,baseline,0,2,54.380287
...,...,...,...,...,...,...,...,...,...,...,...
53995,person_time,impairment,anemia,severe,95_plus,severe,1,baseline,0,9,0.000000
53996,person_time,impairment,anemia,severe,95_plus,severe,2,baseline,0,9,0.000000
53997,person_time,impairment,anemia,severe,95_plus,severe,3,baseline,0,9,0.000000
53998,person_time,impairment,anemia,severe,95_plus,severe,4,baseline,0,9,0.000000


In [6]:
pregnancy_person_time_anemia.groupby("scenario").random_seed.nunique()

scenario
baseline        10
intervention    10
zero            10
Name: random_seed, dtype: int64

In [7]:
pregnancy_person_time_anemia.sub_entity.value_counts()

not_anemic    13500
mild          13500
moderate      13500
severe        13500
Name: sub_entity, dtype: int64

In [8]:
total_pregnant_person_time = aggregate_by_scenario(pregnancy_person_time_anemia)
total_pregnant_person_time

scenario      wealth_quintile
baseline      1                  4.934756e+06
              2                  4.077938e+06
              3                  3.657270e+06
              4                  3.470352e+06
              5                  3.396960e+06
intervention  1                  4.934782e+06
              2                  4.077945e+06
              3                  3.657276e+06
              4                  3.470352e+06
              5                  3.396973e+06
zero          1                  4.934736e+06
              2                  4.077925e+06
              3                  3.657257e+06
              4                  3.470339e+06
              5                  3.396960e+06
Name: value, dtype: float64

In [9]:
anemic_pregnant_person_time = aggregate_by_scenario(
    pregnancy_person_time_anemia[
        pregnancy_person_time_anemia.sub_entity != "not_anemic"
    ]
)
anemic_pregnant_person_time

scenario      wealth_quintile
baseline      1                  2.638684e+06
              2                  2.080855e+06
              3                  1.760789e+06
              4                  1.554804e+06
              5                  1.320301e+06
intervention  1                  2.578060e+06
              2                  2.028059e+06
              3                  1.700731e+06
              4                  1.497855e+06
              5                  1.258039e+06
zero          1                  2.824121e+06
              2                  2.215081e+06
              3                  1.883333e+06
              4                  1.662475e+06
              5                  1.376949e+06
Name: value, dtype: float64

In [10]:
pregnant_anemia_prevalence_by_scenario = (
    anemic_pregnant_person_time / total_pregnant_person_time
).fillna(0)
pregnant_anemia_prevalence_by_scenario

scenario      wealth_quintile
baseline      1                  0.534714
              2                  0.510271
              3                  0.481449
              4                  0.448025
              5                  0.388671
intervention  1                  0.522426
              2                  0.497324
              3                  0.465027
              4                  0.431615
              5                  0.370341
zero          1                  0.572294
              2                  0.543188
              3                  0.514958
              4                  0.479053
              5                  0.405347
Name: value, dtype: float64

In [11]:
path = f"./results/{location}/{vehicle}/pregnant_anemia_prevalence_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
pregnant_anemia_prevalence_by_scenario.to_csv(path)

In [12]:
pop = pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
pop

,sex,age_start,age_end,pregnant,wealth_quintile,value
0,Female,0.0,0.019178,not_pregnant,1,48678.711021
1,Female,0.0,0.019178,not_pregnant,2,42723.422230
2,Female,0.0,0.019178,not_pregnant,3,37932.716572
3,Female,0.0,0.019178,not_pregnant,4,35728.167677
4,Female,0.0,0.019178,not_pregnant,5,28631.476092
...,...,...,...,...,...,...
280,Male,95.0,125.000000,not_pregnant,1,18328.880607
281,Male,95.0,125.000000,not_pregnant,2,19140.234171
282,Male,95.0,125.000000,not_pregnant,3,19770.250303
283,Male,95.0,125.000000,not_pregnant,4,20851.307187


In [13]:
pregnant_pop = pop[pop.pregnant == "pregnant"].groupby(["wealth_quintile"]).value.sum()
pregnant_pop

wealth_quintile
1    4.478246e+06
2    3.707329e+06
3    3.317141e+06
4    3.157155e+06
5    3.080667e+06
Name: value, dtype: float64

In [14]:
pregnancy_prevalent_anemia_cases_by_scenario = (
    pregnant_anemia_prevalence_by_scenario * pregnant_pop
)
pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.394582e+06
              2                  1.891744e+06
              3                  1.597035e+06
              4                  1.414484e+06
              5                  1.197367e+06
intervention  1                  2.339554e+06
              2                  1.843743e+06
              3                  1.542559e+06
              4                  1.362675e+06
              5                  1.140898e+06
zero          1                  2.562875e+06
              2                  2.013778e+06
              3                  1.708188e+06
              4                  1.512444e+06
              5                  1.248741e+06
Name: value, dtype: float64

In [15]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/transition_count_maternal_disorders.parquet"
if pathlib.Path(path).is_file():
    maternal_disorders_transition_counts = pd.read_parquet(path)
else:
    maternal_disorders_transition_counts = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/transition_count_maternal_disorders.parquet"
        ).assign(value=0),
        scenarios,
    )

maternal_disorders_transition_counts

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,1,baseline,0,2,0.0
1,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,2,baseline,0,2,0.0
2,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,3,baseline,0,2,0.0
3,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,4,baseline,0,2,0.0
4,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,5,baseline,0,2,0.0
...,...,...,...,...,...,...,...,...,...,...,...
26995,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,1,baseline,0,9,0.0
26996,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,2,baseline,0,9,0.0
26997,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,3,baseline,0,9,0.0
26998,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,4,baseline,0,9,0.0


In [16]:
maternal_disorders_transition_counts.sub_entity.cat.categories

Index(['susceptible_to_maternal_disorders_to_maternal_disorders', 'maternal_disorders_to_recovered_from_maternal_disorders'], dtype='object')

In [17]:
maternal_disorders_incident_cases_by_scenario = aggregate_by_scenario(
    maternal_disorders_transition_counts[
        maternal_disorders_transition_counts.sub_entity
        == "susceptible_to_maternal_disorders_to_maternal_disorders"
    ]
)
maternal_disorders_incident_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.936750e+06
              2                  1.744958e+06
              3                  2.201745e+06
              4                  1.967388e+06
              5                  1.248686e+06
intervention  1                  2.910972e+06
              2                  1.728365e+06
              3                  2.177025e+06
              4                  1.945409e+06
              5                  1.230747e+06
zero          1                  3.015959e+06
              2                  1.789155e+06
              3                  2.253925e+06
              4                  2.008507e+06
              5                  1.264316e+06
Name: value, dtype: float64

In [18]:
path = (
    f"./results/{location}/{vehicle}/maternal_disorders_incident_cases_by_scenario.csv"
)
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
maternal_disorders_incident_cases_by_scenario.to_csv(path)

In [19]:
path = f"results/rescaled_child_results/{vehicle}/{location}/deaths.parquet"

# NOTE: The child_scenario column currently contains only 'baseline'
# because we didn't have any interventions in the child simulation. If
# we add a child intervention that creates another scenario in this
# column, then results from different child scenarios would get added
# together in the call to aggregate_by_scenario below, so we'd need to
# change the processing code in that case.
def assert_unique_child_scenario(df):
    assert set(df.child_scenario.unique()) == {'baseline'}
    return df

if pathlib.Path(path).is_file():
    neonatal_deaths = (
        pd.read_parquet(path)
        .pipe(assert_unique_child_scenario)
        .rename(columns={"maternal_scenario": "scenario"})
    )
else:
    neonatal_deaths = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/deaths.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_deaths

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,input_draw,random_seed,value
0,deaths,cause,other_causes,other_causes,0_to_5_months,Female,1,baseline,intervention,0,2,7751.180180
1,deaths,cause,other_causes,other_causes,0_to_5_months,Female,2,baseline,intervention,0,2,5777.277153
2,deaths,cause,other_causes,other_causes,0_to_5_months,Female,3,baseline,intervention,0,2,6403.148844
3,deaths,cause,other_causes,other_causes,0_to_5_months,Female,4,baseline,intervention,0,2,5151.405461
4,deaths,cause,other_causes,other_causes,0_to_5_months,Female,5,baseline,intervention,0,2,5055.117509
...,...,...,...,...,...,...,...,...,...,...,...,...
1195,deaths,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,intervention,0,6,866.591573
1196,deaths,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,intervention,0,6,722.159644
1197,deaths,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,intervention,0,6,625.871692
1198,deaths,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,intervention,0,6,433.295786


In [20]:
neonatal_deaths_by_scenario = aggregate_by_scenario(neonatal_deaths)
neonatal_deaths_by_scenario

scenario      wealth_quintile
baseline      1                  199027.197911
              2                  164170.959089
              3                  150161.061994
              4                  137499.196234
              5                  134899.421516
intervention  1                  198882.765982
              2                  163882.095232
              3                  149920.342113
              4                  137162.188401
              5                  134754.989587
zero          1                  199267.917792
              2                  164556.110900
              3                  150305.493923
              4                  137739.916116
              5                  134899.421516
Name: value, dtype: float64

In [21]:
path = f"./results/{location}/{vehicle}/neonatal_deaths_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
neonatal_deaths_by_scenario.to_csv(path)

In [22]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/anemia_cases.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_cases = pd.read_parquet(path)
else:
    non_pregnancy_anemia_cases = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/anemia_cases.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_cases

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,39601.769161,zero
1,Female,0.0,0.019178,2,32639.498130,zero
2,Female,0.0,0.019178,3,28574.791970,zero
3,Female,0.0,0.019178,4,25160.080603,zero
4,Female,0.0,0.019178,5,18957.590376,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,8316.687185,intervention
746,Male,95.0,125.000000,2,7837.566104,intervention
747,Male,95.0,125.000000,3,7803.472372,intervention
748,Male,95.0,125.000000,4,7597.742323,intervention


In [23]:
non_pregnancy_prevalent_anemia_cases_by_scenario = aggregate_by_scenario(
    non_pregnancy_anemia_cases.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  1.230123e+08
              2                  1.156329e+08
              3                  1.141480e+08
              4                  1.087248e+08
              5                  1.032655e+08
intervention  1                  1.190620e+08
              2                  1.115409e+08
              3                  1.096142e+08
              4                  1.040628e+08
              5                  9.759776e+07
zero          1                  1.291309e+08
              2                  1.213341e+08
              3                  1.193293e+08
              4                  1.134219e+08
              5                  1.058060e+08
Name: value, dtype: float64

In [24]:
prevalent_anemia_cases_by_scenario = (
    pregnancy_prevalent_anemia_cases_by_scenario
    + non_pregnancy_prevalent_anemia_cases_by_scenario
)
prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  1.254069e+08
              2                  1.175246e+08
              3                  1.157450e+08
              4                  1.101393e+08
              5                  1.044629e+08
intervention  1                  1.214015e+08
              2                  1.133847e+08
              3                  1.111567e+08
              4                  1.054255e+08
              5                  9.873866e+07
zero          1                  1.316938e+08
              2                  1.233479e+08
              3                  1.210375e+08
              4                  1.149344e+08
              5                  1.070548e+08
Name: value, dtype: float64

In [25]:
path = f"./results/{location}/{vehicle}/prevalent_anemia_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
prevalent_anemia_cases_by_scenario.to_csv(path)

In [26]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
if pathlib.Path(path).is_file():
    ntd_cases_by_scenario = pd.read_csv(path)
else:
    ntd_cases_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/ntd_cases_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

ntd_cases_by_scenario = ntd_cases_by_scenario.set_index(
    ["scenario", "wealth_quintile"]
).value
ntd_cases_by_scenario

scenario      wealth_quintile
zero          1                  4684.597214
              2                  4138.352740
              3                  3715.268143
              4                  3503.314763
              5                  2983.626124
baseline      1                  4479.691264
              2                  3990.943624
              3                  3591.571264
              4                  3395.248207
              5                  2945.009088
intervention  1                  2735.099158
              2                  2574.677328
              3                  2294.381070
              4                  2179.694663
              5                  2117.138152
Name: value, dtype: float64

In [27]:
path = f"./results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)